In [1]:
import pandas as pd
import psycopg2

In [2]:
conn = psycopg2.connect(
    host="localhost",
    database="bank_reviews",
    user="postgres",
    password="E@1992"
)

cursor = conn.cursor()

print("Connected successfully!")

Connected successfully!


In [3]:
df = pd.read_csv(
    "../data/raw/task2_final_results.csv"
)

df.head()

,review_id,review_text,sentiment_label,sentiment_score,identified_theme
0,363a5616-ed3d-4274-85ee-77071067f81d,wow,positive,0.999592,Other
1,56185597-d29b-4a60-a0fb-6783638230a7,Good application,positive,0.999855,Other
2,35efe702-40c9-4e46-ad67-7574ab9ef42d,"Nice, but I can't get some recently transactio...",negative,0.987926,Transaction Performance
3,b41cb49d-59d3-41b8-bbb2-b1952005951b,Very Secure but very poor interface and limite...,negative,0.999777,UI & Design
4,22026bb2-c9c4-4040-892b-2a77ebee48b7,very nice 100%,positive,0.999864,Other


In [5]:
# lets insert bank_id
original_df = pd.read_csv(
    "../data/raw/bank_reviews_cleaned.csv"
)


In [6]:
df["bank"] = original_df["bank"]

In [ ]:
# Create Bank Mapping
bank_mapping = {
    "Commercial Bank of Ethiopia": 1,
    "Bank of Abyssinia": 2,
    "Dashen Bank": 3
}

In [ ]:
# Add bank_id Column
df["bank_id"] = df["bank"].map(
    bank_mapping
)

In [ ]:
# Insert Banks Table
banks_data = [
    (1, "Commercial Bank of Ethiopia", "CBE Mobile Banking"),
    (2, "Bank of Abyssinia", "BOA Mobile"),
    (3, "Dashen Bank", "Dashen Super App")
]

insert_bank_query = """
INSERT INTO banks (
    bank_id,
    bank_name,
    app_name
)
VALUES (%s, %s, %s)
"""

cursor.executemany(
    insert_bank_query,
    banks_data
)

conn.commit()

print("Banks inserted successfully!")

Banks inserted successfully!


In [1]:
import pandas as pd

# Task 1 cleaned scraped data
scraped_df = pd.read_csv(
    "../data/raw/bank_reviews_cleaned.csv"
)

# Task 2 processed data
processed_df = pd.read_csv(
    "../data/raw/task2_final_results.csv"
)

In [2]:
print(scraped_df.columns)

print(processed_df.columns)

Index(['review_id', 'review', 'rating', 'date', 'bank', 'source'], dtype='object')
Index(['review_id', 'review_text', 'sentiment_label', 'sentiment_score',
       'identified_theme'],
      dtype='object')


In [3]:
final_df = pd.merge(
    scraped_df,
    processed_df,
    on="review_id",
    how="inner"
)

In [4]:
bank_mapping = {
    "Commercial Bank of Ethiopia": 1,
    "Bank of Abyssinia": 2,
    "Dashen Bank": 3
}

final_df["bank_id"] = final_df[
    "bank"
].map(bank_mapping)

In [5]:
final_df.rename(
    columns={
        "review": "review_text_original",
        "date": "review_date"
    },
    inplace=True
)

In [7]:
final_df = final_df[
    [
        "review_id",
        "bank_id",
        "review_text",
        "rating",
        "review_date",
        "sentiment_label",
        "sentiment_score",
        "identified_theme",
        "source"
    ]
]

In [8]:
print(final_df.columns)

final_df.head()

Index(['review_id', 'bank_id', 'review_text', 'rating', 'review_date',
       'sentiment_label', 'sentiment_score', 'identified_theme', 'source'],
      dtype='object')


,review_id,bank_id,review_text,rating,review_date,sentiment_label,sentiment_score,identified_theme,source
0,363a5616-ed3d-4274-85ee-77071067f81d,1,wow,5,2026-05-13,positive,0.999592,Other,Google Play
1,56185597-d29b-4a60-a0fb-6783638230a7,1,Good application,2,2026-05-13,positive,0.999855,Other,Google Play
2,35efe702-40c9-4e46-ad67-7574ab9ef42d,1,"Nice, but I can't get some recently transactio...",5,2026-05-13,negative,0.987926,Transaction Performance,Google Play
3,b41cb49d-59d3-41b8-bbb2-b1952005951b,1,Very Secure but very poor interface and limite...,1,2026-05-13,negative,0.999777,UI & Design,Google Play
4,22026bb2-c9c4-4040-892b-2a77ebee48b7,1,very nice 100%,5,2026-05-13,positive,0.999864,Other,Google Play


In [9]:
final_df.to_csv(
    "../data/raw/reviews_final_for_db.csv",
    index=False
)

In [10]:
# Insert Reviews 
insert_review_query = """
INSERT INTO reviews (
    review_id,
    bank_id,
    review_text,
    rating,
    review_date,
    sentiment_label,
    sentiment_score,
    identified_theme,
    source
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

In [16]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="bank_reviews",
    user="postgres",
    password="E@1992"
)

cursor = conn.cursor()

print("Connected successfully!")

Connected successfully!


In [17]:
for _, row in final_df.iterrows():

    cursor.execute(
        insert_review_query,
        (
            str(row["review_id"]),
            int(row["bank_id"]),
            row["review_text"],
            int(row["rating"]),
            row["review_date"],
            row["sentiment_label"],
            float(row["sentiment_score"]),
            row["identified_theme"],
            row["source"]
        )
    )

conn.commit()

print("Reviews inserted successfully!")

Reviews inserted successfully!
